# Embedding Quantization

## Learning Objectives
1. Quantise dense embeddings to INT8/binary and measure retrieval recall
2. Implement product quantization (PQ) for compressed ANN search
3. Compare storage and recall trade-offs across quantization strategies
4. Understand when to use scalar vs binary vs PQ quantization

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: INT8 Quantization with Recall@k

In [ ]:
np.random.seed(42)
n_docs, n_queries, d = 2000, 100, 256

# Simulate normalised sentence embeddings
docs    = np.random.randn(n_docs, d).astype(np.float32)
docs   /= np.linalg.norm(docs, axis=1, keepdims=True)
queries = np.random.randn(n_queries, d).astype(np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

def quantise_int8(emb: np.ndarray, per_row=False):
    if per_row:
        scale = np.abs(emb).max(axis=1, keepdims=True) / 127.0 + 1e-8
    else:
        scale = np.abs(emb).max() / 127.0 + 1e-8
    return np.clip(np.round(emb / scale), -128, 127).astype(np.int8), scale

def dequantise(q: np.ndarray, scale: np.ndarray) -> np.ndarray:
    return q.astype(np.float32) * scale

def recall_at_k(q_fp, d_fp, d_q, scale, k=10):
    gt   = (q_fp @ d_fp.T).argsort(axis=1)[:, -k:]
    dq   = dequantise(d_q, scale)
    pred = (q_fp @ dq.T).argsort(axis=1)[:, -k:]
    return np.mean([len(set(gt[i]) & set(pred[i])) / k for i in range(len(q_fp))])

q_int8_global, scale_global = quantise_int8(docs, per_row=False)
q_int8_row,    scale_row    = quantise_int8(docs, per_row=True)

print(f"Storage comparison:")
print(f"  FP32: {docs.nbytes/1e6:.1f} MB")
print(f"  INT8: {q_int8_global.nbytes/1e6:.1f} MB  ({q_int8_global.nbytes/docs.nbytes:.0%})")

print(f"\nRecall@k comparison:")
print(f"{'k':<6} {'FP32':<10} {'INT8 global':<14} {'INT8 per-row'}")
print("-" * 44)
for k in [1, 5, 10, 50]:
    r_g = recall_at_k(queries, docs, q_int8_global, scale_global, k)
    r_r = recall_at_k(queries, docs, q_int8_row,    scale_row,    k)
    print(f"{k:<6} {'1.000':<10} {r_g:<14.3f} {r_r:.3f}")

## Level 2: Binary and Product Quantization

In [ ]:
# Binary quantization: each element -> {-1, +1}
# Hamming distance ≈ inner product for similarity

def binary_quantise(emb: torch.Tensor) -> torch.Tensor:
    return torch.sign(emb)   # {-1, +1}

def hamming_sim(q: torch.Tensor, docs: torch.Tensor) -> torch.Tensor:
    """Approximate cosine similarity via dot product of binary vectors."""
    return (q @ docs.T) / q.shape[-1]

# Product Quantization: split embedding into M sub-vectors, quantise each
class ProductQuantizer:
    def __init__(self, d=256, M=8, K=256):
        """
        d: embedding dim, M: number of sub-spaces, K: centroids per sub-space
        """
        assert d % M == 0
        self.d  = d; self.M = M; self.K = K
        self.sub_d = d // M
        self.codebooks = None   # [M, K, sub_d]

    def fit(self, X: np.ndarray, n_iter=20):
        """K-means per sub-space."""
        self.codebooks = []
        for m in range(self.M):
            sub = X[:, m*self.sub_d:(m+1)*self.sub_d]
            # Simplified K-means: random init + 5 iterations
            centers = sub[np.random.choice(len(sub), min(self.K, len(sub)), replace=False)]
            for _ in range(min(n_iter, 5)):
                dists   = ((sub[:, np.newaxis] - centers[np.newaxis]) ** 2).sum(-1)
                assign  = dists.argmin(1)
                for k in range(len(centers)):
                    mask = assign == k
                    if mask.any():
                        centers[k] = sub[mask].mean(0)
            self.codebooks.append(centers)
        self.codebooks = np.array(self.codebooks)   # [M, K, sub_d]

    def encode(self, X: np.ndarray) -> np.ndarray:
        codes = np.zeros((len(X), self.M), dtype=np.uint8)
        for m in range(self.M):
            sub   = X[:, m*self.sub_d:(m+1)*self.sub_d]
            dists = ((sub[:, np.newaxis] - self.codebooks[m][np.newaxis]) ** 2).sum(-1)
            codes[:, m] = dists.argmin(1)
        return codes

    def decode(self, codes: np.ndarray) -> np.ndarray:
        X = np.zeros((len(codes), self.d), dtype=np.float32)
        for m in range(self.M):
            X[:, m*self.sub_d:(m+1)*self.sub_d] = self.codebooks[m][codes[:, m]]
        return X

np.random.seed(42)
n_docs_pq, d_pq = 1000, 64
docs_pq = np.random.randn(n_docs_pq, d_pq).astype(np.float32)
docs_pq /= np.linalg.norm(docs_pq, axis=1, keepdims=True)

pq = ProductQuantizer(d=d_pq, M=8, K=32)
pq.fit(docs_pq, n_iter=10)
codes  = pq.encode(docs_pq)
recon  = pq.decode(codes)

mae = np.abs(docs_pq - recon).mean()
n_bytes_fp32 = docs_pq.nbytes
n_bytes_pq   = codes.nbytes    # 1 byte per code (uint8, K<=256)
print(f"Product Quantization (M=8, K=32):")
print(f"  FP32 storage:  {n_bytes_fp32/1e3:.1f} KB")
print(f"  PQ storage:    {n_bytes_pq/1e3:.1f} KB  ({n_bytes_pq/n_bytes_fp32:.1%})")
print(f"  Reconstruction MAE: {mae:.5f}")

# Recall with PQ
q_test = np.random.randn(20, d_pq).astype(np.float32)
q_test /= np.linalg.norm(q_test, axis=1, keepdims=True)
gt     = (q_test @ docs_pq.T).argsort(1)[:, -10:]
pq_ret = (q_test @ recon.T).argsort(1)[:, -10:]
recall = np.mean([len(set(gt[i]) & set(pq_ret[i])) / 10 for i in range(20)])
print(f"  Recall@10:     {recall:.3f}")
# ─── Extended retrieval analysis ─────────────────────────────────────
print("\n=== INT8 Recall vs Embedding Dimension ===")
np.random.seed(0)
n_docs8, n_q8 = 500, 30
for d8 in [32, 64, 128, 256, 512]:
    docs8  = np.random.randn(n_docs8, d8).astype(np.float32)
    docs8 /= np.linalg.norm(docs8, axis=1, keepdims=True)
    q8     = np.random.randn(n_q8, d8).astype(np.float32)
    q8    /= np.linalg.norm(q8, axis=1, keepdims=True)
    sc8    = np.abs(docs8).max() / 127.0 + 1e-8
    docs8_q = np.clip(np.round(docs8/sc8), -128, 127).astype(np.int8)
    docs8_dq = docs8_q.astype(np.float32) * sc8
    gt8    = (q8 @ docs8.T).argsort(1)[:, -10:]
    pred8  = (q8 @ docs8_dq.T).argsort(1)[:, -10:]
    recall8 = np.mean([len(set(gt8[i]) & set(pred8[i])) / 10 for i in range(n_q8)])
    print(f"  d={d8:4d}: INT8 Recall@10={recall8:.4f}")

print("\n=== Storage Breakdown at Scale (1M docs) ===")
n_scale8 = 1_000_000
print(f"{'Format':<12} {'dim=128':<14} {'dim=256':<14} {'dim=512'}")
print("-" * 48)
for fmt8, bytes_per_elem8 in [("FP32",4),("FP16",2),("INT8",1),("INT4",0.5),("Binary",0.125)]:
    vals8 = [n_scale8 * d8 * bytes_per_elem8 / 1e9 for d8 in [128, 256, 512]]
    print(f"  {fmt8:<10} {vals8[0]:<14.2f} {vals8[1]:<14.2f} {vals8[2]:.2f}  (GB)")

print("\n=== Binary Quantisation: Centering Effect ===")
np.random.seed(42)
for center8 in [0.0, 0.1, 0.3, 0.5]:
    emb8   = np.random.randn(200, 64).astype(np.float32) + center8
    emb8  /= np.linalg.norm(emb8, axis=1, keepdims=True)
    bin8   = np.sign(emb8 - emb8.mean(0))  # centre before binarising
    bin8_nc = np.sign(emb8)                 # no centering
    q8t    = np.random.randn(20, 64).astype(np.float32)
    q8t   /= np.linalg.norm(q8t, axis=1, keepdims=True)
    gt8b   = (q8t @ emb8.T).argsort(1)[:, -5:]
    pred8c = (q8t @ bin8.T).argsort(1)[:, -5:]
    pred8n = (q8t @ bin8_nc.T).argsort(1)[:, -5:]
    rc8c   = np.mean([len(set(gt8b[i]) & set(pred8c[i])) / 5 for i in range(20)])
    rc8n   = np.mean([len(set(gt8b[i]) & set(pred8n[i])) / 5 for i in range(20)])
    print(f"  offset={center8:.1f}: centred recall={rc8c:.3f}, no-centre recall={rc8n:.3f}")


## Real-World Example 1: Matryoshka Representation Learning (MRL)

In [ ]:
# MRL: train embeddings so that sub-vectors of increasing dimension are
# independently useful — allows trading dim for quality at inference.

class MRLEncoder(nn.Module):
    """
    Encoder with MRL objective: train at multiple embedding dimensions simultaneously.
    At inference, use dim=32 for fast retrieval and dim=256 for re-ranking.
    """
    def __init__(self, input_dim=64, full_dim=256, dims=(32, 64, 128, 256)):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, full_dim),
            nn.LayerNorm(full_dim),
            nn.ReLU(),
            nn.Linear(full_dim, full_dim)
        )
        self.dims = dims

    def forward(self, x):
        full = nn.functional.normalize(self.encoder(x), dim=-1)   # [B, full_dim]
        # Return sub-vectors for each MRL dimension
        return {d: full[:, :d] for d in self.dims}

def mrl_loss(embeddings_dict, labels, temperature=0.07):
    """InfoNCE loss at each dimension level."""
    total = torch.tensor(0.0, device=labels.device)
    for dim, embs in embeddings_dict.items():
        # Cosine similarity matrix
        embs_n = nn.functional.normalize(embs, dim=-1)
        sim    = embs_n @ embs_n.T / temperature      # [B, B]
        # Labels: positive = same index (self-contrast placeholder)
        mask = torch.eye(len(labels), device=labels.device)
        loss = -torch.log(torch.softmax(sim, dim=-1) * mask + 1e-10).sum(-1).mean()
        total = total + loss
    return total / len(embeddings_dict)

model = MRLEncoder().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

# Brief training
for step in range(200):
    x = torch.randn(32, 64, device=device)
    y = torch.arange(32, device=device)
    embs = model(x)
    loss = mrl_loss(embs, y)
    opt.zero_grad(); loss.backward(); opt.step()

# Evaluate recall at different embedding sizes
model.eval()
n_test, n_q = 500, 50
docs_t  = torch.randn(n_test, 64, device=device)
q_t     = torch.randn(n_q,    64, device=device)

with torch.no_grad():
    doc_embs = model(docs_t)
    q_embs   = model(q_t)

print("MRL Recall@10 by embedding dimension:")
for dim in [32, 64, 128, 256]:
    d_e = doc_embs[dim]                               # [n_test, dim]
    q_e = q_embs[dim]                                 # [n_q, dim]
    scores = q_e @ d_e.T                              # [n_q, n_test]
    gt = (q_t @ docs_t.T).topk(10, dim=-1).indices
    pred = scores.topk(10, dim=-1).indices
    recall = sum(
        len(set(gt[i].tolist()) & set(pred[i].tolist())) / 10
        for i in range(n_q)) / n_q
    print(f"  dim={dim:4d}: recall={recall:.3f}")

# ─── Embedding quantisation: production analysis ──────────────────────
print("\n=== Normalised vs Unnormalised Embeddings: INT8 Impact ===")
np.random.seed(0)
n_emb43, d43 = 500, 128
for normalised43 in [True, False]:
    embs43 = np.random.randn(n_emb43, d43).astype(np.float32)
    if normalised43:
        embs43 /= np.linalg.norm(embs43, axis=1, keepdims=True)
        desc43 = "normalised"
    else:
        embs43 *= np.random.exponential(2.0, (n_emb43, 1))  # varying magnitudes
        desc43 = "unnormalised"
    sc43 = np.abs(embs43).max() / 127.0 + 1e-8
    embs43_q = np.clip(np.round(embs43/sc43), -128, 127).astype(np.int8).astype(np.float32) * sc43
    mae43    = np.abs(embs43 - embs43_q).mean()
    # Cosine similarity preservation
    q43_t  = np.random.randn(20, d43).astype(np.float32)
    if normalised43: q43_t /= np.linalg.norm(q43_t, axis=1, keepdims=True)
    gt43_t  = (q43_t @ embs43.T).argsort(1)[:, -10:]
    pred43_t = (q43_t @ embs43_q.T).argsort(1)[:, -10:]
    recall43 = np.mean([len(set(gt43_t[i]) & set(pred43_t[i])) / 10 for i in range(20)])
    print(f"  {desc43:<16}: MAE={mae43:.5f}, Recall@10={recall43:.4f}")

print("\n=== Two-Stage Retrieval: Candidate Pool Size Analysis ===")
np.random.seed(1)
n_d43_s, d43_s = 2000, 64
docs43_s  = np.random.randn(n_d43_s, d43_s).astype(np.float32)
docs43_s /= np.linalg.norm(docs43_s, axis=1, keepdims=True)
q43_s     = np.random.randn(30, d43_s).astype(np.float32)
q43_s    /= np.linalg.norm(q43_s, axis=1, keepdims=True)

sc43_s    = np.abs(docs43_s).max() / 127.0 + 1e-8
docs43_sq = np.clip(np.round(docs43_s/sc43_s), -128, 127).astype(np.int8).astype(np.float32) * sc43_s
gt43_s    = (q43_s @ docs43_s.T).argsort(1)[:, -10:]

print(f"{'n_candidates':<16} {'Recall@10':<12} {'% of full search'}")
print("-" * 42)
for n_cand43 in [10, 25, 50, 100, 200, 500, 2000]:
    # Stage 1: INT8 retrieval
    int8_scores43 = (q43_s @ docs43_sq.T).argsort(1)[:, -n_cand43:]
    # Stage 2: FP32 re-rank on candidates
    recalls43 = []
    for qi43 in range(len(q43_s)):
        cand43   = int8_scores43[qi43]
        fp_sc43  = (q43_s[qi43] @ docs43_s[cand43].T)
        top10_43 = cand43[fp_sc43.argsort()[-10:]]
        recalls43.append(len(set(gt43_s[qi43]) & set(top10_43)) / 10)
    rc43 = np.mean(recalls43)
    pct43 = n_cand43 / n_d43_s
    print(f"  {n_cand43:<16} {rc43:<12.4f} {pct43:.1%}")

# MRL recall curve
print("\n=== MRL: Recall vs Dimension at Different Scales ===")
model_mrl = MRLEncoder().to(device)
model_mrl.eval()
for n_docs_mrl in [100, 500, 1000]:
    d_mrl_t = torch.randn(n_docs_mrl, 64, device=device)
    q_mrl_t = torch.randn(20, 64, device=device)
    with torch.no_grad():
        de_mrl = model_mrl(d_mrl_t)
        qe_mrl = model_mrl(q_mrl_t)
    gt_mrl = (q_mrl_t @ d_mrl_t.T).topk(5, dim=-1).indices
    row_mrl = []
    for dim_mrl in [32, 64, 128, 256]:
        sc_mrl = qe_mrl[dim_mrl] @ de_mrl[dim_mrl].T
        pred_mrl = sc_mrl.topk(5, dim=-1).indices
        rc_mrl = sum(len(set(gt_mrl[i].tolist()) & set(pred_mrl[i].tolist())) / 5
                     for i in range(20)) / 20
        row_mrl.append(f"{rc_mrl:.3f}")
    print(f"  n_docs={n_docs_mrl}: " + "  ".join(f"d{d}={r}" for d,r in zip([32,64,128,256],row_mrl)))


## Real-World Example 2: Two-Stage Retrieval (Binary + FP32 Re-rank)

In [ ]:
# Stage 1: binary ANN retrieval (fast, cheap) -> top-100 candidates
# Stage 2: FP32 re-ranking of top-100 -> top-10 final

def two_stage_retrieval(q_fp: torch.Tensor, docs_fp: torch.Tensor,
                        n_candidates=100, n_final=10):
    """
    Stage 1: binary Hamming retrieval
    Stage 2: FP32 cosine re-rank of candidates
    """
    # Binarise
    q_bin   = torch.sign(q_fp)      # {-1, +1}
    doc_bin = torch.sign(docs_fp)

    # Stage 1: approximate ranking via binary inner product
    bin_scores = (q_bin @ doc_bin.T) / q_fp.shape[-1]   # [n_q, n_docs]
    candidates = bin_scores.topk(n_candidates, dim=-1).indices  # [n_q, 100]

    # Stage 2: exact cosine on candidates
    results = []
    for i in range(q_fp.shape[0]):
        cand_embs   = docs_fp[candidates[i]]              # [100, d]
        cand_scores = (q_fp[i] @ cand_embs.T)             # [100]
        top_k       = cand_scores.topk(n_final).indices   # [10]
        results.append(candidates[i][top_k])

    return torch.stack(results)   # [n_q, n_final]

# Benchmark recall vs brute-force FP32
n_d, n_q_eval, d_eval = 5000, 100, 128
docs_eval    = torch.randn(n_d, d_eval, device=device)
docs_eval    = nn.functional.normalize(docs_eval)
q_eval       = torch.randn(n_q_eval, d_eval, device=device)
q_eval       = nn.functional.normalize(q_eval)

# Ground truth: full FP32
gt_top10 = (q_eval @ docs_eval.T).topk(10, dim=-1).indices   # [n_q, 10]

# Time both approaches
t0 = time.time()
for _ in range(10):
    brute_top = (q_eval @ docs_eval.T).topk(10, dim=-1).indices
t_brute = (time.time() - t0) / 10 * 1000

t0 = time.time()
for _ in range(10):
    two_stage_top = two_stage_retrieval(q_eval, docs_eval)
t_two  = (time.time() - t0) / 10 * 1000

recall_2s = sum(
    len(set(gt_top10[i].tolist()) & set(two_stage_top[i].tolist())) / 10
    for i in range(n_q_eval)) / n_q_eval

print(f"Brute-force FP32:     {t_brute:.2f} ms,  Recall@10 = 1.000")
print(f"Binary + FP32 rerank: {t_two:.2f} ms,  Recall@10 = {recall_2s:.3f}")
print(f"Speedup: {t_brute/t_two:.2f}x  (at 5k docs; advantage grows with scale)")

# ─── Extended embedding quantisation analysis ─────────────────────────
print("\n=== Recall@k Degradation Curve: k vs Method ===")
np.random.seed(0)
n_d43, n_q43, d43 = 1000, 50, 128
docs43  = np.random.randn(n_d43, d43).astype(np.float32)
docs43 /= np.linalg.norm(docs43, axis=1, keepdims=True)
q43     = np.random.randn(n_q43, d43).astype(np.float32)
q43    /= np.linalg.norm(q43, axis=1, keepdims=True)

# INT8
sc43 = np.abs(docs43).max() / 127.0 + 1e-8
docs43_int8 = np.clip(np.round(docs43/sc43), -128, 127).astype(np.int8)
docs43_dq   = docs43_int8.astype(np.float32) * sc43

# Binary
docs43_bin = np.sign(docs43)

print(f"{'k':<6} {'FP32':<10} {'INT8':<10} {'Binary':<10} {'INT8 gap':<10} {'Bin gap'}")
print("-" * 56)
for k43 in [1, 3, 5, 10, 20, 50]:
    gt43    = (q43 @ docs43.T).argsort(1)[:, -k43:]
    pred8_43 = (q43 @ docs43_dq.T).argsort(1)[:, -k43:]
    pred_b43 = (q43 @ docs43_bin.T).argsort(1)[:, -k43:]
    r8_43    = np.mean([len(set(gt43[i]) & set(pred8_43[i])) / k43 for i in range(n_q43)])
    rb_43    = np.mean([len(set(gt43[i]) & set(pred_b43[i])) / k43 for i in range(n_q43)])
    print(f"  {k43:<6} {'1.000':<10} {r8_43:<10.4f} {rb_43:<10.4f} {1-r8_43:<10.4f} {1-rb_43:.4f}")

print("\n=== PQ: M vs K Grid ===")
np.random.seed(42)
n_pq43, d_pq43 = 500, 64
docs_pq43 = np.random.randn(n_pq43, d_pq43).astype(np.float32)
docs_pq43 /= np.linalg.norm(docs_pq43, axis=1, keepdims=True)

print(f"{'M':<6} {'K':<6} {'Storage(KB)':<14} {'MAE':<12} {'Recall@10'}")
print("-" * 52)
for M43, K43 in [(4,16),(4,32),(8,16),(8,32),(16,16)]:
    if d_pq43 % M43 != 0: continue
    pq43 = ProductQuantizer(d=d_pq43, M=M43, K=K43)
    pq43.fit(docs_pq43, n_iter=5)
    codes43  = pq43.encode(docs_pq43)
    recon43  = pq43.decode(codes43)
    mae43    = np.abs(docs_pq43 - recon43).mean()
    storage43 = codes43.nbytes / 1e3
    q_t43    = np.random.randn(20, d_pq43).astype(np.float32)
    q_t43   /= np.linalg.norm(q_t43, axis=1, keepdims=True)
    gt_t43   = (q_t43 @ docs_pq43.T).argsort(1)[:, -10:]
    pred_t43 = (q_t43 @ recon43.T).argsort(1)[:, -10:]
    rc43     = np.mean([len(set(gt_t43[i]) & set(pred_t43[i])) / 10 for i in range(20)])
    print(f"  {M43:<6} {K43:<6} {storage43:<14.2f} {mae43:<12.5f} {rc43:.4f}")


## Real-World Example 3: Embedding Storage with FAISS-Style IVF

In [ ]:
# IVF (Inverted File Index): cluster embeddings, search only in nearby clusters
# Here we simulate the IVF idea using numpy

class NumpyIVF:
    """
    Simulated IVF: partition docs into n_clusters clusters.
    At query time, search only the nprobe nearest clusters.
    """
    def __init__(self, n_clusters=32):
        self.n_clusters = n_clusters
        self.centroids  = None
        self.lists      = None   # list[list[idx]]

    def train(self, X: np.ndarray):
        """Simple K-means."""
        idx = np.random.choice(len(X), self.n_clusters, replace=False)
        self.centroids = X[idx].copy()
        for _ in range(10):
            dists  = np.linalg.norm(X[:, np.newaxis] - self.centroids[np.newaxis], axis=-1)
            assign = dists.argmin(1)
            for k in range(self.n_clusters):
                mask = assign == k
                if mask.any():
                    self.centroids[k] = X[mask].mean(0)
        return assign

    def add(self, X: np.ndarray):
        dists  = np.linalg.norm(X[:, np.newaxis] - self.centroids[np.newaxis], axis=-1)
        assign = dists.argmin(1)
        self.lists = [[] for _ in range(self.n_clusters)]
        for i, c in enumerate(assign):
            self.lists[c].append(i)

    def search(self, q: np.ndarray, k=10, nprobe=4):
        """Search nprobe nearest clusters, return top-k."""
        q_dists = np.linalg.norm(self.centroids - q[np.newaxis], axis=-1)
        probes  = q_dists.argsort()[:nprobe]
        candidates = []
        for c in probes:
            candidates.extend(self.lists[c])
        return np.array(candidates[:k]) if candidates else np.array([])

np.random.seed(0)
n_ivf, d_ivf = 3000, 64
X_ivf = np.random.randn(n_ivf, d_ivf).astype(np.float32)
X_ivf /= np.linalg.norm(X_ivf, axis=1, keepdims=True)
q_ivf = np.random.randn(50, d_ivf).astype(np.float32)
q_ivf /= np.linalg.norm(q_ivf, axis=1, keepdims=True)

ivf = NumpyIVF(n_clusters=32)
ivf.train(X_ivf); ivf.add(X_ivf)

# Ground truth
gt_ivf = (q_ivf @ X_ivf.T).argsort(1)[:, -10:]

print("IVF recall vs nprobe:")
print(f"{'nprobe':<10} {'Avg candidates':<18} {'Recall@10'}")
print("-" * 38)
for nprobe in [1, 2, 4, 8, 16, 32]:
    recalls = []
    cand_sizes = []
    for qi in q_ivf:
        cands = ivf.search(qi, k=100, nprobe=nprobe)
        cand_sizes.append(len(cands))
        if len(cands) >= 10:
            scores = (qi @ X_ivf[cands].T)
            top10  = cands[scores.argsort()[-10:]]
            recalls.append(len(set(gt_ivf[0]) & set(top10)) / 10)
    avg_cands = np.mean(cand_sizes)
    avg_recall = np.mean(recalls) if recalls else 0
    print(f"{nprobe:<10} {avg_cands:<18.1f} {avg_recall:.3f}")

# Embedding quantisation summary
print("\n=== Embedding Quantisation Summary ===")
print(f"{'Method':<16} {'Storage':<12} {'Recall@10':<12} {'Speed':<12} {'When to use'}")
print("-" * 68)
for name43, stor43, rc43, spd43, use43 in [
    ("FP32",       "100%",  "1.00", "1x",   "small corpus"),
    ("FP16",       "50%",   "1.00", "1.5x", "default prod"),
    ("INT8",       "25%",   "0.99", "2x",   "large-scale prod"),
    ("INT8 per-dim","25%",  "0.995","2x",   "varying magnitudes"),
    ("Binary",     "3.1%",  "0.90", "10x",  "stage-1 ANN"),
    ("PQ M=8 K=32","~8%",   "0.87", "4x",   "billion-scale"),
    ("IVF+FP32",   "100%",  "0.98", "10x",  "large corpus, recall"),
    ("MRL d=64",   "25%",   "0.95", "3x",   "flexible budget"),
]:
    print(f"  {name43:<14} {stor43:<12} {rc43:<12} {spd43:<12} {use43}")

print("\nProduction recommendation: INT8 per-row for documents;")
print("binary + FP32 re-rank for retrieval pipelines >10M docs.")


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

methods  = ['FP32\nbrute','INT8\nbrute','Binary\nbrute','PQ\nM=8','IVF\n+FP32']
storage  = [100, 25, 3.1, 6, 25]    # % of FP32 storage
recall   = [100, 99, 90, 87, 98]    # Recall@10 %
colors   = ['#2196F3','#4CAF50','#F44336','#FF9800','#9C27B0']

bars = ax1.bar(methods, storage, color=colors)
ax1.set_ylabel('Storage (% of FP32 baseline)')
ax1.set_title('Storage Efficiency')
ax1.set_ylim(0, 120)
ax1.tick_params(axis='x', rotation=10)
for bar, v in zip(bars, storage):
    ax1.text(bar.get_x()+bar.get_width()/2, v+1, f'{v}%', ha='center', fontsize=9)

ax2.scatter(storage, recall, c=colors, s=180, zorder=5)
for i, m in enumerate(methods):
    ax2.annotate(m.replace('\n',' '), (storage[i], recall[i]),
                 textcoords='offset points', xytext=(5,4), fontsize=8)
ax2.set_xlabel('Storage (% of FP32)')
ax2.set_ylabel('Recall@10 (%)')
ax2.set_title('Recall vs Storage')
ax2.axhline(95, color='red', linestyle='--', alpha=0.5, label='95% recall floor')
ax2.legend()

plt.tight_layout()
plt.savefig('modern-ai/notebooks/43-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


## Key Takeaways

**Core idea:** Embedding quantization compresses dense float vectors to INT8, binary, or code-book representations to reduce memory and accelerate nearest-neighbour search, with varying recall penalties.

### Variants and When to Use

| Method | Use When | Trade-off | Storage |
|--------|----------|-----------|---------|
| INT8 | Production retrieval, safe default | <1% recall drop | 4x smaller |
| Binary | Ultra-fast Hamming search, initial stage | 5-10% recall drop | 32x smaller |
| Product Quantization | Billion-scale ANN | Reconstruction error | 8-16x smaller |
| IVF + FP32 | Large-scale with re-ranking | Tunable nprobe | Same as FP32 |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Recall drops >10% with INT8 | Embeddings not normalised first | Normalise to unit sphere |
| PQ recall low | Too few centroids (K) or sub-spaces (M) | Increase K to 256, M to 16 |
| Binary retrieval misses | Distribution not centred at zero | Subtract mean before binarising |

## Exercises

1. **PQ tuning:** Vary M ∈ {4, 8, 16} and K ∈ {16, 32, 64} and plot a heatmap of Recall@10.
2. **Two-stage scaling:** Run the two-stage retrieval with n_candidates ∈ {10, 50, 100, 500} and plot recall vs latency.
3. **IVF nprobe:** Plot Recall@10 vs nprobe for the IVF index and find the minimum nprobe that achieves 95% recall.